# Train Support Vector Machine (Classical SVM Pipeline)

End-to-end Classical Machine Learning pipeline on Vietnamese Traffic Signs (VNTS):
- **Features**: Shared classical feature vectors (HOG + LBP + Color Histogram, 2,292-D) standardized on TRAIN only.
- **Model**: Support Vector Classifier (`sklearn.svm.SVC`) with RBF kernel and balanced class weighting.
- **Model Selection**: Cross-Validation hyperparameter tuning optimizes **Best Validation Macro F1** (`scoring: f1_macro`).
- **Artifacts & W&B**: Unified metric logging (`val/macro_f1`, `test/macro_f1`), confusion matrices, and auto-export zip.

Supports running locally or on Kaggle (CPU / GPU environment).


## 0) Bootstrap (Kaggle / Colab)

Run once per session: clones or pulls branch `dl`, changes directory, and installs dependencies.

> **Note for Kaggle**: Remember to toggle **Internet: On** in the notebook sidebar settings.


In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/tuan8p/VN-Traffic-Sign-Classification.git"
REPO_DIR = "VN-Traffic-Sign-Classification"
REPO_BRANCH = "dl"   # <- Unified project branch

base = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
repo_path = base / REPO_DIR

def sh(cmd: str, check: bool = True) -> int:
    print(">>", cmd)
    return subprocess.run(cmd, shell=True, check=check).returncode

if (Path.cwd() / ".git").exists() and (Path.cwd() / "configs" / "shared.yaml").exists():
    repo_path = Path.cwd()
    print("Using the repo this notebook sits in:", repo_path)
elif not (repo_path / ".git").exists():
    sh(f'git clone --branch {REPO_BRANCH} {REPO_URL} "{repo_path}"')

# Ensure on the correct branch and up-to-date
git = f'git -C "{repo_path}"'
sh(f"{git} fetch origin {REPO_BRANCH}")
sh(f"{git} checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}")

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print("cwd:", Path.cwd())
sh(f"{git} log --oneline -1")

# Install dependencies
req = repo_path / "requirements-kaggle.txt"
sh(f'pip install -q -r "{req if req.exists() else repo_path / "requirements.txt"}"')
sh("pip install -q -e .")

# Smoke-test imports
import sklearn
from vn_tsc.pipelines.svm.train import SVMPipeline
from vn_tsc.pipelines.svm.data_adapter import load_svm_data
from vn_tsc.data.dataset import load_class_table
print(f"Bootstrap OK! Scikit-learn: {sklearn.__version__}")


## 1) Point at Processed Data (`data/processed`)

Reads classical features (`features/F_train.npy`, `features/F_val.npy`, `features/F_test.npy`) and class labels.
On Kaggle, attach the dataset from your previous preprocessing run via **+ Add Input**.


In [ ]:
import os
from pathlib import Path

# Search paths for data/processed across Kaggle / Local environments
CANDIDATES = [
    os.environ.get("PROCESSED_ROOT"),
    os.environ.get("SVM_PROCESSED_ROOT"),
    "/kaggle/input/vnts-processed",
    "/kaggle/input/vn-traffic-sign-processed/data/processed",
    "/kaggle/input/vn-tsc-processed/data/processed",
    "/kaggle/input/vietnamese-traffic-signs-processed/data/processed",
    "/kaggle/working/data/processed",
    "data/processed",
]
PROCESSED_ROOT = next((c for c in CANDIDATES if c and (Path(c) / "class_table.csv").exists()), None)

if PROCESSED_ROOT is None:
    print("[!] data/processed/class_table.csv not found in candidate paths.")
    print("    If you haven't run preprocessing yet, run `python -m tools.run_preprocess` first.")
    PROCESSED_ROOT = "data/processed"
else:
    print("[OK] Found processed dataset root at:", PROCESSED_ROOT)
    table = load_class_table(PROCESSED_ROOT)
    print(f"     Total classes: {len(table)} | Mirror pairs: {len(table.mirror_pairs())}")


## 2) SVM Hyperparameters & Overrides

Configure kernel parameters, regularization, dimensionality reduction, and cross-validation tuning below:


In [ ]:
# ==============================================================================
# SVM HYPERPARAMETERS (Edit these variables directly)
# ==============================================================================

# --- 1. Model Architecture & Kernel ---
KERNEL = "rbf"           # "rbf" (default non-linear) or "linear"
C = 1.0                  # Regularization parameter (higher = stricter margin)
GAMMA = "scale"          # Kernel coefficient for 'rbf': 'scale' or 'auto' or float
CLASS_WEIGHT = "balanced"# Adjusts weights inversely proportional to class frequencies
DECISION_SHAPE = "ovo"   # One-vs-one ('ovo') or one-vs-rest ('ovr')

# --- 2. Dimensionality Reduction (Optional) ---
DIM_REDUCTION = "none"   # "none", "pca", or "lda"
PCA_COMPONENTS = 0.95    # Variance ratio to retain if DIM_REDUCTION == 'pca'

# --- 3. Hyperparameter Tuning (Cross-Validation) ---
ENABLE_TUNING = False    # True: runs GridSearchCV on train split before final fit
TUNING_METHOD = "grid"   # "grid" or "random"
CV_FOLDS = 5             # 5-fold Stratified CV
PARAM_GRID = {
    "svm__C": [0.1, 1.0, 10.0],
    "svm__gamma": ["scale", "auto"],
}

# --- 4. Data & Augmentation ---
WITH_AUG = True          # True: includes offline augmented feature vectors (train split only)
PRIMARY_METRIC = "macro_f1" # Locked project primary metric

# --- 5. Logging & Outputs ---
REQUIRE_WANDB = True     # Log metrics and checkpoints to Weights & Biases
EVAL_TEST_AFTER_TRAIN = True # Auto-evaluate on test split upon completion
RUN_NAME = f"svm_{KERNEL}_C{C}_{DIM_REDUCTION}_{'aug' if WITH_AUG else 'noaug'}"


## 3) Resolve Configuration Dict

Combines `configs/pipelines/svm.yaml`, `configs/shared.yaml`, and the hyperparameter overrides above.


In [ ]:
from vn_tsc.config.resolve import resolve_config

OVERRIDES = {
    "model": {
        "kernel": KERNEL,
        "C": C,
        "gamma": GAMMA,
        "class_weight": CLASS_WEIGHT,
        "decision_function_shape": DECISION_SHAPE,
    },
    "dim_reduction": {
        "method": DIM_REDUCTION,
        "pca": {"n_components": PCA_COMPONENTS},
    },
    "tuning": {
        "enabled": ENABLE_TUNING,
        "method": TUNING_METHOD,
        "cv_folds": CV_FOLDS,
        "scoring": "f1_macro",
        "param_grid": PARAM_GRID,
    },
    "train": {
        "with_aug": WITH_AUG,
        "require_wandb": REQUIRE_WANDB,
        "run_name": RUN_NAME,
        "save_model": True,
    },
    "evaluation": {
        "evaluate_test_after_train": EVAL_TEST_AFTER_TRAIN,
    },
    "data": {
        "processed_root": PROCESSED_ROOT,
    },
    "data_source": {
        "processed_root": PROCESSED_ROOT,
    },
}

runtime_yaml = "configs/runtime/kaggle_2xt4.yaml" if Path("/kaggle/working").exists() else "configs/runtime/local.yaml"
cfg = resolve_config(
    pipeline_yaml="configs/pipelines/svm.yaml",
    shared_yaml="configs/shared.yaml",
    runtime_yaml=runtime_yaml,
    overrides=OVERRIDES,
)

print("=== Resolved Configuration ===")
print(f"Run Name       : {RUN_NAME}")
print(f"Pipeline       : {cfg.get('pipeline')}")
print(f"Kernel / C     : {cfg.get('model', {}).get('kernel')} | C={cfg.get('model', {}).get('C')} | gamma={cfg.get('model', {}).get('gamma')}")
print(f"Dim Reduction  : {cfg.get('dim_reduction', {}).get('method')}")
print(f"Tuning (CV)    : {cfg.get('tuning', {}).get('enabled')} (method={cfg.get('tuning', {}).get('method')}, folds={cfg.get('tuning', {}).get('cv_folds')})")
print(f"Primary Metric : {cfg.get('metrics', {}).get('primary', 'macro_f1')}")
print(f"With Aug Data  : {cfg.get('train', {}).get('with_aug')}")
print(f"Processed Root : {cfg.get('data', {}).get('processed_root')}")


## 4) W&B Authentication (if `REQUIRE_WANDB = True`)

On Kaggle: Add your W&B API key under **Add-ons -> Secrets -> `WANDB_API_KEY`** for automatic login.


In [ ]:
if REQUIRE_WANDB:
    import os, wandb
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        wandb_key = user_secrets.get_secret("WANDB_API_KEY")
        wandb.login(key=wandb_key)
        os.environ["WANDB_API_KEY"] = str(wandb_key).strip()
        print("Logged in to W&B via Kaggle User Secrets.")
    except Exception:
        print("Prompting interactive W&B login:")
        wandb.login()
        try:
            s = wandb.setup().settings
            if s and s.api_key:
                os.environ["WANDB_API_KEY"] = str(s.api_key).strip()
        except Exception:
            pass
else:
    import os
    os.environ["WANDB_MODE"] = "offline"
    print("W&B tracking disabled (offline mode).")


## 5) Execute Training

Fits the complete SVM pipeline (StandardScaler + DimReduction + SVC) on the training set and evaluates validation metrics.


In [ ]:
from vn_tsc.runtime.run_dir import make_run_dir
from vn_tsc.utils.seed import set_seed
from vn_tsc.pipelines.svm.train import SVMPipeline

set_seed(int(cfg.get("seed", 42)))
run_dir = make_run_dir(cfg.get("outputs", {}).get("root", "outputs"), "svm", run_name=RUN_NAME)
print(f"Artifacts and checkpoints will be saved to: {run_dir}")

pipeline = SVMPipeline(cfg, run_dir)
train_metrics = pipeline.fit()

print("\n=== Training Finished Successfully ===")
print(f"Model Checkpoint: {run_dir / 'checkpoints' / 'model.joblib'}")
val_metrics = train_metrics.get("validation", {})
print("\nValidation Metrics:")
for k, v in val_metrics.items():
    print(f"  {k:20s}: {v:.4f}" if isinstance(v, float) else f"  {k:20s}: {v}")
if "test" in train_metrics:
    print("\nTest Metrics (Auto-evaluated):\n")
    for k, v in train_metrics["test"].items():
        print(f"  {k:20s}: {v:.4f}" if isinstance(v, float) else f"  {k:20s}: {v}")


## 6) Evaluate on Test Split (Standalone / Re-evaluation)

Evaluates the saved `model.joblib` checkpoint on the held-out test split.


In [ ]:
if "pipeline" not in locals():
    runs = sorted(Path("outputs/svm").glob("*_*"))
    run_dir = runs[-1] if runs else Path("outputs/svm")
    pipeline = SVMPipeline(cfg, run_dir)

cfg["eval"] = {"split": "test"}
test_metrics = pipeline.evaluate()

print("\n=== Test Set Evaluation Results ===")
for k, v in test_metrics.items():
    print(f"  {k:20s}: {v:.4f}" if isinstance(v, float) else f"  {k:20s}: {v}")


## 7) Visualize Training & Evaluation Artifacts

Displays the confusion matrix figures generated for validation and test splits.


In [ ]:
from IPython.display import Image, display

if "run_dir" not in locals() or not run_dir.exists():
    runs = sorted(Path("outputs/svm").glob("*_*"))
    run_dir = runs[-1] if runs else Path("outputs/svm")

print("Visualizing artifacts from:", run_dir)
figs_to_display = [
    (run_dir / "figures" / "confusion_matrix_val.png", "Confusion Matrix on Validation Split"),
    (run_dir / "figures" / "confusion_matrix_test.png", "Confusion Matrix on Test Split"),
]

for path, title in figs_to_display:
    if path.exists():
        print(f"\n>>> {title}:")
        display(Image(filename=str(path)))
    else:
        print(f"Note: {path.name} not found at {path}")


## 8) Breakdown: Rare Classes & Small Object Analysis

Inspects performance and details on rare traffic sign classes (< 30 samples).


In [ ]:
import json
from vn_tsc.data.dataset import load_class_table, rare_classes

if "run_dir" not in locals() or not run_dir.exists():
    runs = sorted(Path("outputs/svm").glob("*_*"))
    run_dir = runs[-1] if runs else Path("outputs/svm")

table = load_class_table(cfg["data"]["processed_root"])
rare_ids = rare_classes(cfg["data"]["processed_root"], threshold=30)

print(f"Total dataset classes  : {len(table)}")
print(f"Rare classes (<30 crops): {len(rare_ids)} classes")
for cid in rare_ids[:10]:
    print(f"  Class {cid:02d} ({table[cid].sign_code:10s}): {table[cid].name_vi}")
if len(rare_ids) > 10:
    print(f"  ... and {len(rare_ids) - 10} more rare classes")

metrics_file = run_dir / "metrics.json"
if metrics_file.exists():
    with open(metrics_file, "r", encoding="utf-8") as f:
        all_metrics = json.load(f)
    print("\nSummary metrics stored in metrics.json:")
    print(json.dumps(all_metrics, indent=2))
